In [1]:
df = spark.read.format('delta').load('Tables/dbo/silver_sales')

print(f'Row Count : {df.count()}')
print(f'Columns : {df.columns}')
display(df.limit(10))

StatementMeta(, c3565b29-e7ce-44aa-ad04-a8747ad4b1c8, 3, Finished, Available, Finished, False)

Row Count : 1268
Columns : ['order_id', 'order_date', 'customer_name', 'region', 'product_category', 'revenue', 'quantity', 'status', 'revenue_usd']


SynapseWidget(Synapse.DataFrame, dd4aa2a1-33ff-4310-88aa-94b63b93c841)

In [2]:
from pyspark.sql.functions import col, date_format, count, sum, avg, round

df_dated = df.withColumn(
    'year_month',
    date_format(col('order_date'), 'yyyy-MM')
)
display(df_dated.limit(10))


StatementMeta(, c3565b29-e7ce-44aa-ad04-a8747ad4b1c8, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b50ca003-1ed7-4483-8510-381b160e8fcc)

In [3]:
df_gold = df_dated.groupBy('region', 'year_month') \
                .agg(
                    round(sum('revenue'),2).alias('total_revenue'),
                    count('order_id').alias('total_orders'),
                    round(avg('revenue')).alias('average_revenue')
                ) \
                .orderBy('year_month', 'region')
print(f'Gold Table Row Count : {df_gold.count()}')
display(df_gold.limit(10))


StatementMeta(, c3565b29-e7ce-44aa-ad04-a8747ad4b1c8, 5, Finished, Available, Finished, False)

Gold Table Row Count : 96


SynapseWidget(Synapse.DataFrame, 9d9fe829-0486-40b6-b0e9-37087a5fb38f)

In [4]:
df_gold.write.format('delta').mode('overwrite').saveAsTable('gold_sales_summary')

StatementMeta(, c3565b29-e7ce-44aa-ad04-a8747ad4b1c8, 6, Finished, Available, Finished, False)